# WP46 — Corrigibility Under Self-Improvement

**Prometheus v0 PoC · Work Package 46**

> *"The machine would have to be given some goal or goals … and also the ability to modify its own goals if it discovered that they were somehow mistaken."*
> — I.J. Good, *Speculations Concerning the First Ultraintelligent Machine*, 1965

## What WP46 does

Good's corrigibility requirement: a human must be able to **interrupt**, **correct**, and **safely resume** an ultraintelligent machine mid-run — without catastrophic loss of state or unsafe side-effects.

WP46 demonstrates this on the 14-layer CRLS stack:

1. **OverrideQueue** — receives human override signals at specified generations.
2. **CorrigibleSimulator** — polls the queue each generation; on override: saves checkpoint, applies parameters, logs audit record, resumes.
3. **CorrigibilityReport** — proves state is preserved and accuracy recovers post-interrupt.

Two scheduled overrides:
- **Gen 30**: Raise safety threshold (0.60 → 0.75) — *safety review*
- **Gen 70**: Reset strategy probs to uniform — *curriculum shift*


In [ ]:
import sys, os, pathlib

def _find_repo_root() -> str:
    """Return the Prometheus_v0_PoC repo root regardless of kernel CWD."""
    candidates = [
        # Running from inside notebooks/
        os.path.abspath(os.path.join(os.getcwd(), "..")),
        # Running from repo root
        os.getcwd(),
        # Absolute fallback — known install location
        str(pathlib.Path.home() / "Prometheus_v0_PoC"),
        "/home/pmc/Prometheus_v0_PoC",
    ]
    for c in candidates:
        if os.path.isdir(os.path.join(c, "prometheus")):
            return c
    raise RuntimeError(
        "Cannot locate Prometheus_v0_PoC repo root. "
        f"Tried: {candidates}"
    )

repo_root = _find_repo_root()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import logging
logging.basicConfig(level=logging.WARNING)

from prometheus.wp46_corrigibility import (
    run_corrigibility_demo,
    verify_wp46_exit_criteria,
    OverrideSignal,
    OverrideQueue,
    CorrigibleSimulator,
)
print("WP46 loaded ✓")

## Run the Corrigibility Demo

In [ ]:
report = run_corrigibility_demo(n_generations=120, seed=42)
print(report.summary())


## Accuracy Curve with Interrupt Markers

Does the system recover after each human override?

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.use("Agg")
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

accs = report.accuracies
n = len(accs)

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(range(n), accs, "b-", linewidth=1.5, label="Accuracy", alpha=0.85)

    colors = ["red", "orange", "purple"]
    for i, irec in enumerate(report.interrupt_records):
        g   = irec.generation
        col = colors[i % len(colors)]
        ax.axvline(g, color=col, linestyle="--", alpha=0.8, linewidth=1.5,
                   label=f"Override {i+1} (gen {g}): {irec.reason[:35]}…")
        # Pre/post windows
        pre_start  = max(0, g - 8)
        post_end   = min(n, g + 8)
        ax.axvspan(pre_start, g,        alpha=0.07, color=col)
        ax.axvspan(g, post_end,         alpha=0.12, color=col)
        ax.text(g + 1, min(accs) + 0.01, f"Override {i+1}", color=col, fontsize=8, rotation=90)

    ax.set_xlabel("Generation")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"Corrigibility Demo — {report.n_interrupts} Overrides  |  State preserved: {report.state_preserved}")
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(True, alpha=0.3)
    ax.set_ylim(max(0, min(accs) - 0.02), min(1, max(accs) + 0.02))

    plt.tight_layout()
    plt.savefig("wp46_corrigibility.png", dpi=100)
    plt.show()
    print("Plot saved → wp46_corrigibility.png")
else:
    print("matplotlib not available")
    for i, irec in enumerate(report.interrupt_records):
        print(f"Override {i+1} at gen {irec.generation}: {irec.reason}")
        print(f"  Pre-acc: {report.pre_interrupt_acc[i]:.4f}  Post-acc: {report.post_interrupt_acc[i]:.4f}  Recovery: {report.recovery_speed[i]:.1f} gens")


## Audit Trail

Every interrupt is logged with a full `InterruptRecord`.

In [ ]:
import json
print("Interrupt records (audit trail):")
print()
for irec in report.interrupt_records:
    print(json.dumps(irec.to_dict(), indent=2))
    print()


## Checkpoint Inspection

In [ ]:
print("Saved checkpoints:")
print()
for ck in report.checkpoints:
    print(json.dumps(ck.to_dict(), indent=2))
    print()


## Good's Verdict

In [ ]:
print(report.good_verdict)


## WP46 Exit Criteria

In [ ]:
criteria = verify_wp46_exit_criteria(report)
all_pass = all(criteria.values())
print(f"{'PASS' if all_pass else 'FAIL'} — WP46 Exit Criteria")
print()
for name, result in criteria.items():
    status = "✓" if result else "✗"
    print(f"  [{status}] {name}")
print()
print(f"All criteria pass: {all_pass}")


## Conclusion

WP46 demonstrates that Prometheus satisfies Good's **corrigibility** requirement:

1. **Mid-run overrides** are accepted without crashing or losing state.
2. **Checkpoint save/restore** guarantees that accuracy does not fall below 90% of the pre-interrupt level.
3. **Audit trail** logs every interrupt with timestamp, reason, parameters changed, and checkpoint reference.
4. **Fast recovery**: accuracy returns to pre-interrupt levels within ≤ 15 generations.

**Russell (2019)**: The system *welcomes* correction as information about human preferences — not as an obstacle to avoid.
